In [887]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import numpy as np

In [888]:
bus = pd.read_csv('Nodes/Bus.csv')
fgc = pd.read_csv('Nodes/FGC.csv')
metro = pd.read_csv('Nodes/Metro.csv')
tram = pd.read_csv('Nodes/Tram.csv')

all_stops = pd.concat([ bus, fgc, metro, tram], ignore_index=True)
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")

exchange_edges = pd.read_csv('Edges/Exchanges.csv')
exchange_edges

,origen,dest,tram,mode,lines,type,time,directed,geometry
0,SB-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - 59,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864)
1,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864)
2,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864)
3,SB-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - H16,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864)
4,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864)
...,...,...,...,...,...,...,...,...,...
138110,B-V33-3596,T-T6-MINA,La Mina - Ponent - La Mina,Bus - Tram,V33 - T6,Exchange,04:34,True,"LINESTRING (2.2172843 41.4188065, 2.2174015 41..."
138111,B-B23-105999,T-T6-MINA,Rambla de la Mina - La Mina,Bus - Tram,B23 - T6,Exchange,01:48,True,"LINESTRING (2.2224278 41.4182191, 2.222341 41...."
138112,B-B23-106000,T-T6-MINA,CAP La Mina - La Mina,Bus - Tram,B23 - T6,Exchange,04:10,True,"LINESTRING (2.2201838 41.4156256, 2.2204138 41..."
138113,T-T5-PARC,T-T6-MINA,Parc del Besòs - La Mina,Tram - Tram,T5 - T6,Exchange,04:51,True,"LINESTRING (2.2171297 41.4191608, 2.2171685 41..."


In [889]:
exchange_edges['mode'].value_counts()

mode
Bus - Bus        127624
Metro - Bus        4341
Tram - Bus         1684
Bus - Tram         1543
Bus - Metro        1103
FGC - Bus           727
Bus - FGC           607
Tram - Tram         177
Metro - Metro        91
Metro - FGC          81
FGC - FGC            71
Metro - Tram         47
Tram - Metro         14
FGC - Metro           5
Name: count, dtype: int64

In [890]:
exchange_edges = exchange_edges.merge(all_stops[['id','stop_id']], left_on='dest', right_on='id', how='left')
exchange_edges

,origen,dest,tram,mode,lines,type,time,directed,geometry,id,stop_id
0,SB-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - 59,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
1,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
2,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
3,SB-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - H16,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864),B-H16-2,2
4,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,01:00,True,POINT (2.198984998801282 41.393104003754864),B-H16-2,2
...,...,...,...,...,...,...,...,...,...,...,...
138110,B-V33-3596,T-T6-MINA,La Mina - Ponent - La Mina,Bus - Tram,V33 - T6,Exchange,04:34,True,"LINESTRING (2.2172843 41.4188065, 2.2174015 41...",T-T6-MINA,MINA
138111,B-B23-105999,T-T6-MINA,Rambla de la Mina - La Mina,Bus - Tram,B23 - T6,Exchange,01:48,True,"LINESTRING (2.2224278 41.4182191, 2.222341 41....",T-T6-MINA,MINA
138112,B-B23-106000,T-T6-MINA,CAP La Mina - La Mina,Bus - Tram,B23 - T6,Exchange,04:10,True,"LINESTRING (2.2201838 41.4156256, 2.2204138 41...",T-T6-MINA,MINA
138113,T-T5-PARC,T-T6-MINA,Parc del Besòs - La Mina,Tram - Tram,T5 - T6,Exchange,04:51,True,"LINESTRING (2.2171297 41.4191608, 2.2171685 41...",T-T6-MINA,MINA


# Tram

In [891]:
exchange_edges_tram = exchange_edges[exchange_edges['dest'].str.startswith('T')]

In [892]:
stop_times_x = pd.read_table('Data/Tram/TBX/stop_times.txt', sep=',')
stop_times_S = pd.read_table('Data/Tram/TBS/stop_times.txt', sep=',')
stop_times = pd.concat([stop_times_x, stop_times_S])    
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [893]:
stops_x = pd.read_table('Data/Tram/TBX/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops_s = pd.read_table('Data/Tram/TBS/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops = pd.concat([stops_x, stops_s])
stops = stops[~stops['stop_id'].str.startswith('S')]
stops['stop_id'] = stops['stop_id'].astype(int)

In [894]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [895]:
trips_x = pd.read_table('Data/Tram/TBX/trips.txt', sep=',')
trips_s = pd.read_table('Data/Tram/TBS/trips.txt', sep=',')
trips = pd.concat([trips_x, trips_s])
trips = trips[['route_id','trip_id']]

In [896]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','route_id','departure_time'], keep='first', inplace=True)
times_w_routes['route_id'] = 'T' + times_w_routes['route_id'].astype(int).astype(str)
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].str[2:]
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].replace({'RIGL':'SRMN','LLEV':'BDOR','MRSM':'DGMR','CATA':'CTLN','JOAN':'STJB','MRTI':'STMR','SROC':'STRC'})

In [897]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['stop_id', 'route_id', 'stop_name', 'stop_desc'], as_index=False)['wait_time']
    .mean()
 )
avg_wait_times = avg_wait_times[['stop_desc', 'route_id', 'wait_time']]
avg_wait_times['id'] = 'T' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_desc']
avg_wait_times = avg_wait_times[['id', 'wait_time']]

In [898]:
exchange_edges_tram = exchange_edges_tram.merge(avg_wait_times, left_on='dest', right_on='id', how='left')
exchange_edges_tram = exchange_edges_tram[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]


# FCG

In [899]:
exchange_edges_fgc = exchange_edges[exchange_edges['dest'].str.startswith('F')]

In [900]:
stop_times = pd.read_table('Data/FGC/gtfs_fgc/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [901]:
stops = pd.read_table('Data/FGC/gtfs_fgc/stops.txt', sep=',')[['stop_id','stop_name']]

In [902]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [903]:
trips = pd.read_table('Data/FGC/gtfs_fgc/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
trips = trips[['route_id','trip_id','trip_headsign']]
valid = ['L6', 'L7', 'L8', 'L12','S1']
trips = trips[trips['route_id'].isin(valid)]

In [904]:
times_w_routes = trips.merge(stops_w_times, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['route_id','stop_name','departure_time','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['departure_time'].notna()]
times_w_routes['stop_id']  = times_w_routes['stop_id'].str[:2]

In [905]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)
avg_wait_times = avg_wait_times.groupby(['route_id', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'M' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_id']
avg_wait_times

,route_id,stop_id,stop_name,wait_time
0,L12,M-L12-RE,Reina Elisenda,1.602361
1,L12,M-L12-SR,Sarrià,1.602361
2,L6,M-L6-BN,La Bonanova,1.667934
3,L6,M-L6-GR,Gràcia,1.434656
4,L6,M-L6-MN,Muntaner,1.669772
5,L6,M-L6-PC,Barcelona - Plaça Catalunya,1.435007
6,L6,M-L6-PR,Provença,1.408198
7,L6,M-L6-SG,Sant Gervasi,1.665641
8,L6,M-L6-SR,Sarrià,1.698788
9,L6,M-L6-TT,Les Tres Torres,1.685035


In [906]:
exchange_edges_fgc = exchange_edges_fgc.merge(avg_wait_times[['stop_id','wait_time']], left_on='dest', right_on='stop_id', how='left')
exchange_edges_fgc = exchange_edges_fgc[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]

# Bus

In [907]:
exchange_edges_bus = exchange_edges[exchange_edges['dest'].str.startswith('B')]

## AMB

In [908]:
stop_times = pd.read_table('Data/GTFS_AMB/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id', 'stop_id', 'departure_time']].dropna()

stop_times['departure_time'] = pd.to_datetime(
    stop_times['departure_time'],
    format='%H:%M:%S',
    errors='coerce'
)

stop_times = stop_times[stop_times['departure_time'].dt.hour.between(7, 11)]

In [909]:
stops = pd.read_table('Data/GTFS_AMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

,stop_id,stop_name
0,109303,Eusebi Güell - Joaquim Auger
1,100005,Escola Busquets i Punset
2,109239,Faigs - Montseny
3,109244,Faigs - Freixe
4,1461,Av de Cornellà - Pont d'Esplugues
...,...,...
4922,100031,Palau Reial
4923,100032,Zona Universitària
4924,112179,Pl. Jacint Verdaguer
4925,112099,Dr. Robert - Av. Bufalà


In [910]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [911]:
trips = pd.read_table('Data/GTFS_AMB/trips.txt', sep=',')
trips = trips[['route_id','trip_id','trip_headsign']]

In [912]:
routes = pd.read_table('Data/GTFS_AMB/routes.txt', sep=',')[['route_id','route_short_name']] 

In [913]:
times_w_routes_amb = trips.merge(stops_w_times, on='trip_id', how='left')
times_w_routes_amb = routes.merge(times_w_routes_amb, on='route_id', how='left')
times_w_routes_amb = times_w_routes_amb[times_w_routes_amb['departure_time'].notna()]
times_w_routes_amb.drop_duplicates(subset=['route_id','stop_name','departure_time'], keep='first', inplace=True)
times_w_routes_amb  =  times_w_routes_amb['stop_id'].astype(int).astype(str)

## TMB

In [914]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]
stop_times

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_69472/2680156778.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]


,trip_id,stop_id,departure_time
56,1.104.11029978,1.936,07:00:15
57,1.104.11029978,1.935,07:02:25
58,1.104.11029978,1.934,07:04:00
59,1.104.11029978,1.933,07:06:15
60,1.104.11029978,1.932,07:08:43
...,...,...,...
1827601,2.250.102.3185372.3051,2.3690.694679,09:59:00
1827606,2.250.102.3185372.3051,2.1257.700582,10:11:00
1827614,2.250.102.3185372.3051,2.784.674381,10:23:00
1827623,2.250.102.3185372.3051,2.1887.685159,10:38:00


In [915]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

,stop_id,stop_name
0,1.111,Hospital de Bellvitge
1,E.1011101,Ascensor - Residència sanitària
2,E.11101,Residència sanitària
3,P.6660111,Hospital de Bellvitge
4,1.112,Bellvitge
...,...,...
3465,2.9979.699905,"Meridiana, 298"
3466,2.9980.699903,"Meridiana, 320"
3467,2.9981.699910,"Concepción Arenal, 103-105"
3468,2.9994.700010,"Torrent de la Perera, 23"


In [916]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,1.111,Hospital de Bellvitge,1.1.11924033,07:44:45
1,1.111,Hospital de Bellvitge,1.1.11924034,07:49:13
2,1.111,Hospital de Bellvitge,1.1.11924035,09:29:15
3,1.111,Hospital de Bellvitge,1.1.11924036,09:32:09
4,1.111,Hospital de Bellvitge,1.1.11924059,07:53:05
...,...,...,...,...
82048,2.9980.699903,"Meridiana, 320",2.229.149.3364059.3024,07:24:00
82049,2.9980.699903,"Meridiana, 320",2.229.149.3364062.3024,09:13:00
82050,2.9980.699903,"Meridiana, 320",2.229.149.3364080.3024,07:42:00
82051,2.9980.699903,"Meridiana, 320",2.229.149.3364082.3024,09:27:00


In [917]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5','L9','L10','L11','L9N','L9S','L10N','L10S','FM','M1','M9','978']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[~trips['route_short_name'].isin(metro)]
trips

,route_id,route_short_name,trip_id,trip_headsign
40199,2.220.2999,D20,2.220.64.3369726.2999,Ernest Lluch
40200,2.220.2999,D20,2.220.64.3369728.2999,Ernest Lluch
40201,2.220.2999,D20,2.220.64.3369724.2999,Ernest Lluch
40202,2.220.2999,D20,2.220.64.3369708.2999,Ernest Lluch
40203,2.220.2999,D20,2.220.64.3369678.2999,Ernest Lluch
...,...,...,...,...
80906,2.196.2971,196,2.196.26.3064023.2971,Av. Tibidabo
80907,2.196.2971,196,2.196.26.3063966.2971,Av. Tibidabo
80908,2.196.2971,196,2.196.26.3064008.2971,Av. Tibidabo
80909,2.196.2971,196,2.196.26.3063968.2971,Av. Tibidabo


In [918]:
times_w_routes_tmb = trips.merge(stops_w_times, on='trip_id', how='left')
times_w_routes_tmb = times_w_routes_tmb[times_w_routes_tmb['departure_time'].notna()]
times_w_routes_tmb.drop_duplicates(subset=['route_id','stop_name','departure_time'], keep='first', inplace=True)
times_w_routes_tmb['stop_id'] = times_w_routes_tmb['stop_id'].str.split('.').str[1]
times_w_routes_tmb

,route_id,route_short_name,trip_id,trip_headsign,stop_id,stop_name,departure_time
2,2.220.2999,D20,2.220.64.3369724.2999,Ernest Lluch,1788,Trelawny - Av Litoral - Final de trajecte,10:54:00
5,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1164,Pla de Palau - Pl Pau Vila,08:35:00
6,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1185,Paral·lel - Av Mistral,08:55:00
7,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1326,Paral·lel - Drassanes,08:43:00
8,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1713,Les Corts - Aurora Bertrana,09:09:00
...,...,...,...,...,...,...,...
82185,2.196.2971,196,2.196.26.3050863.2971,Av. Tibidabo,3453,Av Tibidabo - La Rotonda - Final de trajecte,08:49:00
82186,2.196.2971,196,2.196.26.3050863.2971,Av. Tibidabo,3474,Av Tibidabo - Pg Sant Gervasi,08:44:00
82268,2.196.2971,196,2.196.26.3063944.2971,Av. Tibidabo,3453,Av Tibidabo - La Rotonda - Final de trajecte,10:20:00
82292,2.196.2971,196,2.196.26.3063946.2971,Av. Tibidabo,3453,Av Tibidabo - La Rotonda - Final de trajecte,10:50:00


## Together

In [919]:
times_w_routes = pd.concat([times_w_routes_tmb, times_w_routes_amb], ignore_index=True)
times_w_routes

,route_id,route_short_name,trip_id,trip_headsign,stop_id,stop_name,departure_time,0
0,2.220.2999,D20,2.220.64.3369724.2999,Ernest Lluch,1788,Trelawny - Av Litoral - Final de trajecte,10:54:00,NaN
1,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1164,Pla de Palau - Pl Pau Vila,08:35:00,NaN
2,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1185,Paral·lel - Av Mistral,08:55:00,NaN
3,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1326,Paral·lel - Drassanes,08:43:00,NaN
4,2.220.2999,D20,2.220.64.3369656.2999,Ernest Lluch,1713,Les Corts - Aurora Bertrana,09:09:00,NaN
...,...,...,...,...,...,...,...,...
223433,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106867
223434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3123
223435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109921
223436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112662


In [920]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'trip_headsign','departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )
times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id','route_short_name', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)

avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'B' + '-' + avg_wait_times['route_short_name'] + '-' + avg_wait_times['stop_id']
avg_wait_times

,route_id,route_short_name,stop_id,stop_name,wait_time
0,2.102.2876,102,B-102-1801,Agrupació número 16·17,2.500000
1,2.104.2887,104,B-104-1801,Agrupació número 16·17,1.500000
2,2.107.2933,107,B-107-1400,Can Tunis,18.375000
3,2.107.2933,107,B-107-2318,Sagrat Cor,18.000000
4,2.107.2933,107,B-107-2691,Via Nostra - Sra de la Merçé,17.500000
...,...,...,...,...,...
895,2.96.2873,96,B-96-3776,Bach - Joan Miró - Final de trajecte,4.614583
896,2.96.2873,96,B-96-3952,"av. Ribera, 13-54",4.520000
897,2.96.2873,96,B-96-9968,"Juan de Garay, 116",4.515993
898,2.97.2874,97,B-97-2718,Pl Primer de Maig - Final de trajecte,4.770833


In [921]:
exchange_edges_bus = exchange_edges_bus.merge(avg_wait_times[['stop_id','wait_time']], left_on='dest', right_on='stop_id', how='left')
exchange_edges_bus = exchange_edges_bus[['origen', 'dest', 'tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]
median = exchange_edges_bus['wait_time'].median()
print(exchange_edges_bus['wait_time'].mean())
exchange_edges_bus['wait_time'] = exchange_edges_bus['wait_time'].fillna(median)
print(exchange_edges_bus['wait_time'].mean())

4.275450299329422
2.85010930223154


# Metro

In [922]:
exchange_edges_metro = exchange_edges[exchange_edges['dest'].str.startswith('M')]
exchange_edges_metro['lines'].value_counts()

lines
IU Stop - L3    53
IU Stop - L1    50
IU Stop - L4    49
IU Stop - L2    23
IU Stop - L5    23
                ..
LH1 - L5         1
V25 - L5         1
H2 - L5          1
T1 - L5          1
104 - L11        1
Name: count, Length: 254, dtype: int64

In [923]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 8]
stop_times

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_69472/2171042604.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]


,trip_id,stop_id,departure_time
56,1.104.11029978,1.936,07:00:15
57,1.104.11029978,1.935,07:02:25
58,1.104.11029978,1.934,07:04:00
59,1.104.11029978,1.933,07:06:15
60,1.104.11029978,1.932,07:08:43
...,...,...,...
1826691,2.250.102.3185348.3051,2.3690.694679,07:13:00
1826696,2.250.102.3185348.3051,2.1257.700582,07:21:00
1826704,2.250.102.3185348.3051,2.784.674381,07:30:00
1826713,2.250.102.3185348.3051,2.1887.685159,07:42:00


In [924]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

,stop_id,stop_name
0,1.111,Hospital de Bellvitge
1,E.1011101,Ascensor - Residència sanitària
2,E.11101,Residència sanitària
3,P.6660111,Hospital de Bellvitge
4,1.112,Bellvitge
...,...,...
3465,2.9979.699905,"Meridiana, 298"
3466,2.9980.699903,"Meridiana, 320"
3467,2.9981.699910,"Concepción Arenal, 103-105"
3468,2.9994.700010,"Torrent de la Perera, 23"


In [925]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,1.111,Hospital de Bellvitge,1.1.11924033,07:44:45
1,1.111,Hospital de Bellvitge,1.1.11924034,07:49:13
2,1.111,Hospital de Bellvitge,1.1.11924059,07:53:05
3,1.111,Hospital de Bellvitge,1.1.11924060,07:57:33
4,1.111,Hospital de Bellvitge,1.1.11924250,07:01:51
...,...,...,...,...
20351,2.9980.699903,"Meridiana, 320",2.229.115.3335909.3024,07:47:00
20352,2.9980.699903,"Meridiana, 320",2.229.149.3363784.3024,07:59:00
20353,2.9980.699903,"Meridiana, 320",2.229.149.3363847.3024,07:06:00
20354,2.9980.699903,"Meridiana, 320",2.229.149.3364059.3024,07:24:00


In [926]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5','L9','L10','L11']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[trips['route_short_name'].isin(metro)]
trips

,route_id,route_short_name,trip_id,trip_headsign
0,1.1.1,L1,1.1.11781120,Hospital de Bellvitge
1,1.1.1,L1,1.1.11781121,Fondo
2,1.1.1,L1,1.1.11781113,Fondo
3,1.1.1,L1,1.1.11781114,Hospital de Bellvitge
4,1.1.1,L1,1.1.11781115,Fondo
...,...,...,...,...
40186,1.11.1,L11,1.11.11820563,Trinitat Nova
40187,1.11.1,L11,1.11.11820566,Can Cuiàs
40188,1.11.1,L11,1.11.11820570,Can Cuiàs
40189,1.11.1,L11,1.11.11820573,Trinitat Nova


In [927]:
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,1.111,Hospital de Bellvitge,1.1.11924033,07:44:45
1,1.111,Hospital de Bellvitge,1.1.11924034,07:49:13
2,1.111,Hospital de Bellvitge,1.1.11924059,07:53:05
3,1.111,Hospital de Bellvitge,1.1.11924060,07:57:33
4,1.111,Hospital de Bellvitge,1.1.11924250,07:01:51
...,...,...,...,...
20351,2.9980.699903,"Meridiana, 320",2.229.115.3335909.3024,07:47:00
20352,2.9980.699903,"Meridiana, 320",2.229.149.3363784.3024,07:59:00
20353,2.9980.699903,"Meridiana, 320",2.229.149.3363847.3024,07:06:00
20354,2.9980.699903,"Meridiana, 320",2.229.149.3364059.3024,07:24:00


In [928]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','departure_time','route_id','route_short_name','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['route_id'].notna()]
times_w_routes

,stop_id,stop_name,trip_id,departure_time,route_id,route_short_name,trip_headsign
0,1.111,Hospital de Bellvitge,1.1.11924033,07:44:45,1.1.1,L1,Hospital de Bellvitge
1,1.111,Hospital de Bellvitge,1.1.11924034,07:49:13,1.1.1,L1,Fondo
2,1.111,Hospital de Bellvitge,1.1.11924059,07:53:05,1.1.1,L1,Hospital de Bellvitge
3,1.111,Hospital de Bellvitge,1.1.11924060,07:57:33,1.1.1,L1,Fondo
4,1.111,Hospital de Bellvitge,1.1.11924250,07:01:51,1.1.1,L1,Hospital de Bellvitge
...,...,...,...,...,...,...,...
6228,1.555,Ernest Lluch,1.5.11925831,07:07:05,1.5.1,L5,Vall d'Hebron
6229,1.555,Ernest Lluch,1.5.11925859,07:15:30,1.5.1,L5,Vall d'Hebron
6230,1.555,Ernest Lluch,1.5.11925904,07:23:55,1.5.1,L5,Vall d'Hebron
6231,1.555,Ernest Lluch,1.5.11925931,07:03:15,1.5.1,L5,Cornellà Centre


In [929]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','route_short_name','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id','route_short_name', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)
avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times

,route_id,route_short_name,stop_id,stop_name,wait_time
0,1.1.1,L1,1.111,Hospital de Bellvitge,4.218056
1,1.1.1,L1,1.112,Bellvitge,4.210714
2,1.1.1,L1,1.113,Av. Carrilet,4.223611
3,1.1.1,L1,1.114,Rambla Just Oliveras,4.223611
4,1.1.1,L1,1.115,Can Serra,4.223611
...,...,...,...,...,...
118,1.5.1,L5,1.531,Horta,0.567448
119,1.5.1,L5,1.532,El Carmel,0.554407
120,1.5.1,L5,1.533,El Coll | La Teixonera,0.558194
121,1.5.1,L5,1.534,Vall d'Hebron,0.555369


In [930]:
exchange_edges_metro['dest_name'] = exchange_edges_metro['tram'].str.split(' - ').str[-1]
exchange_edges_metro['dest_name'] = exchange_edges_metro['dest_name'].replace({'Av. de Xile':'Ernest Lluch'})

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_69472/2383961833.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exchange_edges_metro['dest_name'] = exchange_edges_metro['tram'].str.split(' - ').str[-1]
/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_69472/2383961833.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exchange_edges_metro['dest_name'] = exchange_edges_metro['dest_name'].replace({'Av. de Xile':'Ernest Lluch'})


In [931]:
exchange_edges_metro = exchange_edges_metro.merge(avg_wait_times[['stop_name','wait_time']], left_on='dest_name', right_on='stop_name', how='left')
exchange_edges_metro = exchange_edges_metro[['origen', 'dest','dest_name','tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]
exchange_edges_metro

,origen,dest,dest_name,tram,mode,lines,type,time,wait_time,directed,geometry
0,SM-121,M-L1-121,Hostafrancs,Hostafrancs - Hostafrancs,Metro - Metro,IU Stop - L1,Exchange - Self,03:00,4.166667,True,POINT (2.14329127 41.37525363)
1,SM-122,M-L1-122,Espanya,Espanya - Espanya,Metro - Metro,IU Stop - L1,Exchange - Self,03:00,4.166667,True,POINT (2.149387 41.37545911)
2,SM-122,M-L1-122,Espanya,Espanya - Espanya,Metro - Metro,IU Stop - L1,Exchange - Self,03:00,1.505060,True,POINT (2.149387 41.37545911)
3,M-L3-122,M-L1-122,Espanya,Espanya - Espanya,Metro - Metro,L3 - L1,Exchange - Self,02:00,4.166667,True,POINT (2.149387 41.37545911)
4,M-L3-122,M-L1-122,Espanya,Espanya - Espanya,Metro - Metro,L3 - L1,Exchange - Self,02:00,1.505060,True,POINT (2.149387 41.37545911)
...,...,...,...,...,...,...,...,...,...,...,...
1691,B-76-1733,M-L11-1139,Ciutat Meridiana,Les Agudes 118-124 - Ciutat Meridiana,Bus - Metro,76 - L11,Exchange,03:23,NaN,True,"LINESTRING (2.1739342 41.4611568, 2.1740398 41..."
1692,B-183-1733,M-L11-1139,Ciutat Meridiana,Les Agudes 118-124 - Ciutat Meridiana,Bus - Metro,183 - L11,Exchange,03:23,NaN,True,"LINESTRING (2.1739342 41.4611568, 2.1740398 41..."
1693,B-62-2610,M-L11-1139,Ciutat Meridiana,Les Agudes 140 - Ciutat Meridiana,Bus - Metro,62 - L11,Exchange,03:58,NaN,True,"LINESTRING (2.1717241 41.4603543, 2.1717024 41..."
1694,B-76-2610,M-L11-1139,Ciutat Meridiana,Les Agudes 140 - Ciutat Meridiana,Bus - Metro,76 - L11,Exchange,03:58,NaN,True,"LINESTRING (2.1717241 41.4603543, 2.1717024 41..."


In [932]:
exchange_edges_metro[exchange_edges_metro['wait_time'].isna()]

,origen,dest,dest_name,tram,mode,lines,type,time,wait_time,directed,geometry
83,SM-910,M-L9-910,Mercabarna,Mercabarna - Mercabarna,Metro - Metro,IU Stop - L9,Exchange - Self,03:00,NaN,True,POINT (2.11125716 41.33350192)
84,SM-911,M-L9-911,Parc Logístic,Parc Logístic - Parc Logístic,Metro - Metro,IU Stop - L9,Exchange - Self,03:00,NaN,True,POINT (2.12740125 41.34166355)
85,SM-932,M-L9-932,Onze de Setembre,Onze de Setembre - Onze de Setembre,Metro - Metro,IU Stop - L9,Exchange - Self,03:00,NaN,True,POINT (2.19361021 41.42961749)
86,M-L10-932,M-L9-932,Onze de Setembre,Onze de Setembre - Onze de Setembre,Metro - Metro,L10 - L9,Exchange - Self,02:00,NaN,True,POINT (2.19361021 41.42961749)
87,SM-932,M-L10-932,Onze de Setembre,Onze de Setembre - Onze de Setembre,Metro - Metro,IU Stop - L10,Exchange - Self,03:00,NaN,True,POINT (2.19361021 41.42961749)
...,...,...,...,...,...,...,...,...,...,...,...
1691,B-76-1733,M-L11-1139,Ciutat Meridiana,Les Agudes 118-124 - Ciutat Meridiana,Bus - Metro,76 - L11,Exchange,03:23,NaN,True,"LINESTRING (2.1739342 41.4611568, 2.1740398 41..."
1692,B-183-1733,M-L11-1139,Ciutat Meridiana,Les Agudes 118-124 - Ciutat Meridiana,Bus - Metro,183 - L11,Exchange,03:23,NaN,True,"LINESTRING (2.1739342 41.4611568, 2.1740398 41..."
1693,B-62-2610,M-L11-1139,Ciutat Meridiana,Les Agudes 140 - Ciutat Meridiana,Bus - Metro,62 - L11,Exchange,03:58,NaN,True,"LINESTRING (2.1717241 41.4603543, 2.1717024 41..."
1694,B-76-2610,M-L11-1139,Ciutat Meridiana,Les Agudes 140 - Ciutat Meridiana,Bus - Metro,76 - L11,Exchange,03:58,NaN,True,"LINESTRING (2.1717241 41.4603543, 2.1717024 41..."


# Save

In [662]:
exchanges = pd.concat([exchange_edges_tram, exchange_edges_fgc, exchange_edges_bus, exchange_edges_metro], ignore_index=True)
exchanges.to_csv('Edges/Exchanges_with_Wait_Times.csv', index=False)

NameError: name 'exchange_edges_metro' is not defined